# Incident Response Assistant (LangGraph HITL)

This notebook builds an incident workflow that:
1. Collects logs and context
2. Summarizes incident impact and likely cause
3. Proposes next response steps
4. Pauses for human approval before final report


## Workflow-First Explanation

The design goal is safe automation: automate analysis, keep final approval with an engineer.

```text
START
  -> collect_logs_context
  -> summarize_incident
  -> propose_next_steps
  -> [PAUSE before finalize_response]
  -> finalize_response
  -> END
```


## Node Design and State Passing

Each node has one responsibility and writes explicit state fields.

- `collect_logs_context`: writes `collected_logs`, `service_context`
- `summarize_incident`: writes `incident_summary`, `severity`, `suspected_root_cause`
- `propose_next_steps`: writes `proposed_actions`
- `finalize_response`: writes `final_response` using human approval notes

State is the shared contract between nodes. Each downstream node reads upstream fields instead of re-computing them.


## Step 1 - Install and Import

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph'])
print('langgraph installed')


langgraph installed


In [2]:
from typing import List, TypedDict
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver

print('Imports ready')


E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Imports ready


## Step 2 - Define Incident State

In [3]:
class IncidentState(TypedDict):
    alert_text: str
    collected_logs: List[str]
    service_context: str
    incident_summary: str
    severity: str
    suspected_root_cause: str
    proposed_actions: List[str]
    approval_decision: str
    approval_notes: str
    final_response: str

print('IncidentState defined')


IncidentState defined


## Step 3 - Sample Alert and Observability Data

In [4]:
SAMPLE_ALERT = (
    'ALERT: API error rate spike to 38% for gateway-prod in us-east-1; '
    'customer checkout failures observed in last 12 minutes.'
)

SIMULATED_LOGS = [
    '14:03:11 gateway-prod 502 upstream connect error to user-service',
    '14:03:14 user-service pod restart count increased to 6',
    '14:03:19 user-service memory pressure warning at 96%',
    '14:03:22 order-service timeout calling user-service profile endpoint',
    '14:03:26 deploy event detected for user-service at 13:41 UTC',
    '14:03:31 gateway-prod checkout endpoint p95 latency crossed 4200ms',
]

SERVICE_CONTEXT = (
    'gateway-prod depends on user-service for auth profile hydration. '
    'order-service and checkout flow call gateway-prod. '
    'User-service recently received a caching patch in latest deploy.'
)

print('Alert and simulated logs loaded')
print('Log lines:', len(SIMULATED_LOGS))


Alert and simulated logs loaded
Log lines: 6


## Step 4 - Node: Collect Logs and Context

In [5]:
def collect_logs_context(state: IncidentState) -> dict:
    print('Node collect_logs_context: collecting observability artifacts')
    return {'collected_logs': SIMULATED_LOGS, 'service_context': SERVICE_CONTEXT}


## Step 5 - Node: Summarize Incident

In [6]:
def summarize_incident(state: IncidentState) -> dict:
    logs_joined = ' | '.join(state['collected_logs']).lower()

    if 'checkout failures' in state['alert_text'].lower() and '502' in logs_joined:
        severity = 'SEV-1'
    elif 'error rate spike' in state['alert_text'].lower():
        severity = 'SEV-2'
    else:
        severity = 'SEV-3'

    if 'memory pressure' in logs_joined and 'restart count' in logs_joined:
        root_cause = 'User-service instability likely driven by memory pressure after recent deploy.'
    elif 'timeout' in logs_joined:
        root_cause = 'Upstream dependency timeout likely causing gateway failures.'
    else:
        root_cause = 'Root cause unclear; requires deeper service tracing.'

    summary = (
        'Incident Summary: gateway-prod is returning elevated 502 errors, '
        'impacting checkout flows and downstream order operations. '
        'Cross-service symptoms indicate degradation centered on user-service.'
    )

    print('Node summarize_incident: severity', severity)
    return {'incident_summary': summary, 'severity': severity, 'suspected_root_cause': root_cause}


## Step 6 - Node: Propose Next Steps

In [7]:
def propose_next_steps(state: IncidentState) -> dict:
    actions = [
        'Mitigate immediately: scale user-service replicas and raise memory limits for hot path pods.',
        'Stabilize traffic: enable gateway circuit-breaker fallback for profile hydration dependency.',
        'Contain blast radius: rollback recent user-service caching patch if error budget burn continues.',
        'Coordinate response: open incident channel and assign incident commander + communications owner.',
        'Verification: track checkout success rate, 5xx error rate, and p95 latency every 5 minutes.',
    ]

    if state['severity'] == 'SEV-1':
        actions.insert(0, 'Escalate immediately to on-call lead and platform manager due to customer impact.')

    print('Node propose_next_steps: proposed', len(actions), 'actions')
    return {'proposed_actions': actions}


## Step 7 - Node: Finalize Response After Approval

In [8]:
def finalize_response(state: IncidentState) -> dict:
    lines = []
    lines.append('INCIDENT RESPONSE REPORT')
    lines.append('Severity: ' + state['severity'])
    lines.append('Summary: ' + state['incident_summary'])
    lines.append('Suspected root cause: ' + state['suspected_root_cause'])
    lines.append('Approval decision: ' + state['approval_decision'])
    lines.append('Approval notes: ' + state['approval_notes'])
    lines.append('Approved next steps:')
    for step in state['proposed_actions']:
        lines.append('- ' + step)

    report = '\n'.join(lines)
    print('Node finalize_response: report assembled')
    return {'final_response': report}


## Step 8 - Build Graph and Configure Pause for Approval

In [9]:
builder = StateGraph(IncidentState)
builder.add_node('collect_logs_context', collect_logs_context)
builder.add_node('summarize_incident', summarize_incident)
builder.add_node('propose_next_steps', propose_next_steps)
builder.add_node('finalize_response', finalize_response)

builder.add_edge(START, 'collect_logs_context')
builder.add_edge('collect_logs_context', 'summarize_incident')
builder.add_edge('summarize_incident', 'propose_next_steps')
builder.add_edge('propose_next_steps', 'finalize_response')
builder.add_edge('finalize_response', END)

checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer, interrupt_before=['finalize_response'])

print('Graph compiled with HITL pause before finalize_response')


Graph compiled with HITL pause before finalize_response


## Pause/Approval Mechanism Explained

- First invoke runs until pause point and returns intermediate state.
- Engineer reviews summary and actions.
- Engineer decision is injected into saved state with `update_state`.
- Second invoke resumes from pause and completes final report generation.


## Step 9 - Run Phase 1 (Collect -> Summarize -> Propose -> Pause)

In [10]:
config = {'configurable': {'thread_id': 'incident-34-thread-1'}}
initial_state = {
    'alert_text': SAMPLE_ALERT,
    'collected_logs': [],
    'service_context': '',
    'incident_summary': '',
    'severity': '',
    'suspected_root_cause': '',
    'proposed_actions': [],
    'approval_decision': '',
    'approval_notes': '',
    'final_response': '',
}

phase1 = app.invoke(initial_state, config)

print('Workflow paused before finalize_response')
print('Severity:', phase1['severity'])
print('Summary:', phase1['incident_summary'])
print('Root cause:', phase1['suspected_root_cause'])
print('Proposed actions:')
for i, action in enumerate(phase1['proposed_actions'], start=1):
    print(str(i) + '. ' + action)


Node collect_logs_context: collecting observability artifacts
Node summarize_incident: severity SEV-1
Node propose_next_steps: proposed 6 actions
Workflow paused before finalize_response
Severity: SEV-1
Summary: Incident Summary: gateway-prod is returning elevated 502 errors, impacting checkout flows and downstream order operations. Cross-service symptoms indicate degradation centered on user-service.
Root cause: User-service instability likely driven by memory pressure after recent deploy.
Proposed actions:
1. Escalate immediately to on-call lead and platform manager due to customer impact.
2. Mitigate immediately: scale user-service replicas and raise memory limits for hot path pods.
3. Stabilize traffic: enable gateway circuit-breaker fallback for profile hydration dependency.
4. Contain blast radius: rollback recent user-service caching patch if error budget burn continues.
5. Coordinate response: open incident channel and assign incident commander + communications owner.
6. Verifi

## Step 10 - Simulate Human Approval and Resume

In [11]:
approval_decision = 'approved_with_changes'
approval_notes = (
    'Approve actions with priority on rollback readiness and customer communication every 15 minutes. '
    'Keep gateway fallback enabled until 5xx remains below threshold for 30 minutes.'
)

app.update_state(config, {'approval_decision': approval_decision, 'approval_notes': approval_notes})
phase2 = app.invoke(None, config)

print('Workflow resumed and completed')
print('\n' + phase2['final_response'])


Node finalize_response: report assembled
Workflow resumed and completed

INCIDENT RESPONSE REPORT
Severity: SEV-1
Summary: Incident Summary: gateway-prod is returning elevated 502 errors, impacting checkout flows and downstream order operations. Cross-service symptoms indicate degradation centered on user-service.
Suspected root cause: User-service instability likely driven by memory pressure after recent deploy.
Approval decision: approved_with_changes
Approval notes: Approve actions with priority on rollback readiness and customer communication every 15 minutes. Keep gateway fallback enabled until 5xx remains below threshold for 30 minutes.
Approved next steps:
- Escalate immediately to on-call lead and platform manager due to customer impact.
- Mitigate immediately: scale user-service replicas and raise memory limits for hot path pods.
- Stabilize traffic: enable gateway circuit-breaker fallback for profile hydration dependency.
- Contain blast radius: rollback recent user-service c

## Step 11 - Checkpoint History

In [12]:
history = list(app.get_state_history(config))
print('Checkpoint count:', len(history))
for idx, snapshot in enumerate(history):
    source = 'init'
    if snapshot.metadata and 'source' in snapshot.metadata:
        source = snapshot.metadata['source']
    populated = [k for k, v in snapshot.values.items() if v]
    print(f'Checkpoint {idx}: source={source}, populated_fields={populated}')


Checkpoint count: 7
Checkpoint 0: source=loop, populated_fields=['alert_text', 'collected_logs', 'service_context', 'incident_summary', 'severity', 'suspected_root_cause', 'proposed_actions', 'approval_decision', 'approval_notes', 'final_response']
Checkpoint 1: source=update, populated_fields=['alert_text', 'collected_logs', 'service_context', 'incident_summary', 'severity', 'suspected_root_cause', 'proposed_actions', 'approval_decision', 'approval_notes']
Checkpoint 2: source=loop, populated_fields=['alert_text', 'collected_logs', 'service_context', 'incident_summary', 'severity', 'suspected_root_cause', 'proposed_actions']
Checkpoint 3: source=loop, populated_fields=['alert_text', 'collected_logs', 'service_context', 'incident_summary', 'severity', 'suspected_root_cause']
Checkpoint 4: source=loop, populated_fields=['alert_text', 'collected_logs', 'service_context']
Checkpoint 5: source=loop, populated_fields=['alert_text']
Checkpoint 6: source=input, populated_fields=[]


## Final Notes

This pattern is useful for operational safety:
- automate diagnosis and drafting
- require explicit human approval before high-impact actions
- preserve full state history for auditability
